<a href="https://colab.research.google.com/github/giyuubin/group-project/blob/main/static_scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pefile shap

In [3]:
import pefile
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import warnings
import sys
warnings.filterwarnings('ignore')

print("🛡️ [Phase 4] LightGBM 실전 정적 스캐너 (Static Scanner) 기동...\n")

# ==========================================
# 1. 파일 경로 세팅 (내 환경에 맞게 수정!)
# ==========================================
MODEL_PATH = 'lightgbm_final.pkl'
DATA_PATH = 'optimized_data_test.csv'  # 피처명 50개를 추출하기 위한 원본 데이터
TARGET_EXE = 'malware_test.exe'        # MalwareBazaar에서 다운받은 실제 악성코드!

# ==========================================
# 2. 모델 및 피처 목록 로드
# ==========================================
try:
    model = joblib.load(MODEL_PATH)
    # 데이터의 첫 줄(컬럼명)만 읽어와서 50개의 타겟 피처 리스트를 만듭니다.
    df_sample = pd.read_csv(DATA_PATH, nrows=0)
    target_features = [col for col in df_sample.columns if col != 'malware']
    print(f"✅ 모델 및 타겟 피처({len(target_features)}개) 로드 완료")
except Exception as e:
    print(f"❌ 초기화 에러: {e}")
    sys.exit()

# ==========================================
# 3. PE 파일 정적 분석 및 API 추출 함수
# ==========================================
def extract_apis_from_exe(exe_path):
    print(f"🔍 [{exe_path}] 정적 분석 중... (실행하지 않으므로 안전합니다)")
    api_set = set()
    try:
        pe = pefile.PE(exe_path)
        if hasattr(pe, 'DIRECTORY_ENTRY_IMPORT'):
            for entry in pe.DIRECTORY_ENTRY_IMPORT:
                for imp in entry.imports:
                    if imp.name is not None:
                        # API 이름을 문자열로 디코딩하여 저장
                        api_name = imp.name.decode('utf-8', 'ignore')
                        api_set.add(api_name)
        return api_set
    except Exception as e:
        print(f"❌ PE 파일 파싱 에러 (잘못된 파일이거나 난독화됨): {e}")
        return set()

# ==========================================
# 4. 바이너리 인코딩 (50차원 벡터 변환)
# ==========================================
# 실제 악성코드에서 뽑아낸 API 목록
extracted_apis = extract_apis_from_exe(TARGET_EXE)

# 타겟 피처 50개와 비교하여 [1, 0, 0, 1...] 형태의 딕셔너리로 만듭니다.
vector_dict = {}
for feature in target_features:
    if feature in extracted_apis:
        vector_dict[feature] = 1
    else:
        vector_dict[feature] = 0

# 모델에 넣기 위해 1행짜리 DataFrame으로 변환
df_input = pd.DataFrame([vector_dict])
print(f"✅ 벡터 변환 완료: 발견된 타겟 피처 {sum(vector_dict.values())}개 / 50개")

# ==========================================
# 5. 모델 추론 (Inference) 및 차단 판별
# ==========================================
THRESHOLD = 0.85 # 우정님이 정하신 최적의 타협점

pred_prob = model.predict_proba(df_input)[0][1]

print("\n" + "="*50)
if pred_prob >= THRESHOLD:
    print(f"🚨 [경고] 악성코드 탐지!! (확률: {pred_prob:.2%})")
    print(f"🚨 차단 기준치({THRESHOLD*100}%)를 초과하여 프로세스 실행을 격리합니다.")
else:
    print(f"✅ [안전] 정상 파일입니다. (악성 확률: {pred_prob:.2%})")
print("="*50 + "\n")

# ==========================================
# 6. SHAP XAI (결정 근거 시각화)
# ==========================================
print("🧠 SHAP 결정 근거(Waterfall Plot) 생성 중...")

explainer = shap.TreeExplainer(model)
shap_values = explainer(df_input)

# 클래스 1(악성)에 대한 SHAP 값 추출
if len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[0], max_display=10, show=False)
plt.title(f"XAI Static Analysis Result\nPredicted Probability: {pred_prob:.2%} (Threshold: {THRESHOLD})", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('demo_shap_waterfall.png', dpi=300, bbox_inches='tight')
plt.show()

print("🎯 시연용 파이프라인 구동 완료! (이미지 저장됨: demo_shap_waterfall.png)")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



🛡️ [Phase 4] LightGBM 실전 정적 스캐너 (Static Scanner) 기동...

❌ 초기화 에러: [Errno 2] No such file or directory: 'lightgbm_final.pkl'
Traceback (most recent call last):
  File "/tmp/ipykernel_1554/893813374.py", line 24, in <cell line: 0>
    model = joblib.load(MODEL_PATH)
            ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/joblib/numpy_pickle.py", line 735, in load
    with open(filename, "rb") as f:
         ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'lightgbm_final.pkl'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1554/893813374.py", line 31, in <cell line: 0>
    sys.exit()
SystemExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):

TypeError: object of type 'NoneType' has no len()